# Giai đoạn 2 — Làm sạch & chuẩn hóa.

Giai đoạn 2 — Làm sạch & chuẩn hóa.
Input : datas/hour.csv (raw)
Output: datas/02_cleaned/cleaned.csv + dtypes.json + cleaning_report.json
- dteday str -> datetime, sort theo (dteday, hr)
- 8 cột rời rạc -> category (GIỮ category, không astype int trở lại)
- Không xóa dòng (raw không null); chỉ báo cáo null/duplicates/outlier-nhẹ
Chạy: python src/02_clean.py

In [1]:
import json
from pathlib import Path

import pandas as pd
# Cấu hình đường dẫn (inline để notebook chạy độc lập, không cần config.py)
ROOT = Path.cwd()
if not (ROOT / "datas").exists() and (ROOT.parent / "datas").exists():
    ROOT = ROOT.parent  # khi kernel chạy từ trong thư mục src/
RAW_CSV = ROOT / "datas" / "hour.csv"
EDA_DIR = ROOT / "datas" / "01_eda"
CLEANED_DIR = ROOT / "datas" / "02_cleaned"
FEATURES_DIR = ROOT / "datas" / "03_features"
SPLIT_DIR = ROOT / "datas" / "04_split"
MODELS_DIR = ROOT / "datas" / "05_models"
EVAL_DIR = ROOT / "datas" / "06_evaluation"
CAT_COLS = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "time_period"]
TEST_SIZE = 0.2
RANDOM_STATE = 42

CAT8 = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]


In [2]:
def main():
    CLEANED_DIR.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(RAW_CSV)
    report = {"n_raw": len(df)}

    df["dteday"] = pd.to_datetime(df["dteday"])
    for c in CAT8:
        if c in df.columns:
            df[c] = df[c].astype("category")

    df = df.sort_values(["dteday", "hr"]).reset_index(drop=True)
    report.update({
        "n_cleaned": len(df),
        "null_total": int(df.isnull().sum().sum()),
        "duplicates": int(df.duplicated().sum()),
        "date_min": str(df["dteday"].min()),
        "date_max": str(df["dteday"].max()),
    })

    df.to_csv(CLEANED_DIR / "cleaned.csv", index=False)
    (CLEANED_DIR / "dtypes.json").write_text(
        df.dtypes.astype(str).to_json(indent=2), encoding="utf-8")
    (CLEANED_DIR / "cleaning_report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(report, indent=2, ensure_ascii=False))
    print(f"OK -> {CLEANED_DIR / 'cleaned.csv'}")


In [3]:
if __name__ == "__main__":
    main()


{
  "n_raw": 17379,
  "n_cleaned": 17379,
  "null_total": 0,
  "duplicates": 0,
  "date_min": "2011-01-01 00:00:00",
  "date_max": "2012-12-31 00:00:00"
}
OK -> /mnt/d/Documents/UIT/HK2/CKIE313/datas/02_cleaned/cleaned.csv
